# 07 — Preparação para Modelagem

## Objetivo

Este notebook tem como objetivo preparar a base analítica enriquecida para a etapa de modelagem supervisionada do Tech Challenge — Fase 3.

A partir das evidências e recomendações produzidas durante a Análise Exploratória de Dados (EDA), serão consolidadas as decisões necessárias para definir as variáveis que poderão participar do processo de modelagem.

Nesta etapa, será adotado um cenário de predição preventiva, utilizando apenas informações que poderiam estar disponíveis antes da aplicação da avaliação de 2025.

Com isso, o notebook deverá estabelecer uma base segura e metodologicamente consistente para a construção posterior da pipeline de Machine Learning.

## 1. Carregamento da base analítica final

A preparação para modelagem parte da base analítica final construída após as etapas de enriquecimento.

Antes de aplicar qualquer seleção ou transformação, é necessário carregar esse artefato e confirmar sua estrutura básica, garantindo que a etapa de modelagem seja iniciada a partir da base esperada.

In [0]:
# Objetivo:
#
# Carregar a base analítica final produzida
# após as etapas de enriquecimento.
#
# Justificativa:
#
# A preparação para modelagem deve partir
# do artefato consolidado e validado nas
# etapas anteriores do projeto.
#
# A conferência inicial de sua dimensão
# permite verificar se a quantidade de
# registros e colunas permanece conforme
# o esperado.
#
# Ação:
#
# Carrega a base analítica final e exibe
# sua dimensão.

import pandas as pd

FASE2_ROOT = (
    "/Volumes/workspace/default/vol_trio_drive/"
    "projetos/fiap/tech_challenge_fase2"
)

FASE2_GOLD_PATH = f"{FASE2_ROOT}/gold"

INPUT_PATH = (
    f"{FASE2_GOLD_PATH}/"
    "alunos_base_enriquecida_finalizada/"
    "base_analitica_final.csv"
)


FASE3_ROOT = (
    "/Volumes/workspace/default/vol_trio_drive/"
    "projetos/fiap/tech_challenge_fase3"
)

MODELAGEM_PATH = f"{FASE3_ROOT}/modelagem"


dbutils.fs.mkdirs(MODELAGEM_PATH)


base_modelagem = pd.read_csv(
    INPUT_PATH,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

base_modelagem.shape

## 2. Validação estrutural da base

Após confirmar a dimensão da base analítica final, o próximo passo consiste em verificar sua estrutura.

Essa validação permite confirmar os nomes das variáveis e seus respectivos tipos de dados antes de iniciar a definição das features que poderão participar da modelagem.

In [0]:
# Objetivo:
#
# Verificar a estrutura da base carregada para
# a etapa de preparação da modelagem.
#
# Justificativa:
#
# Antes de definir as variáveis preditoras,
# é necessário confirmar os nomes das colunas
# e seus respectivos tipos de dados.
#
# Essa conferência também permite identificar
# eventuais divergências em relação à estrutura
# consolidada ao final do enriquecimento.
#
# Ação:
#
# Exibe as informações estruturais da base,
# incluindo colunas, tipos de dados e valores
# não nulos.

base_modelagem.info()

## 3. Definição do problema preditivo

O objetivo da modelagem é prever se um aluno será considerado alfabetizado ou não alfabetizado, utilizando informações educacionais, territoriais e socioeconômicas disponíveis antes da aplicação da avaliação de 2025.

A variável-alvo do problema é `in_alfabetizado`, definida na base original da avaliação.

### Momento da predição

Será adotado um cenário de predição preventiva, no qual o modelo deverá estimar o risco de não alfabetização antes da aplicação da avaliação de 2025 e, consequentemente, antes do conhecimento de seus resultados.

Essa definição estabelece a principal regra para a seleção das variáveis preditoras:

> uma informação somente poderá participar do modelo se estiver disponível no momento em que a predição seria realizada.

Variáveis produzidas durante ou após a aplicação da avaliação deverão, portanto, ser analisadas quanto ao risco de *data leakage* antes da definição final das features.

## 4. Auditoria das variáveis para modelagem

Com o problema preditivo e o momento da predição definidos, o próximo passo consiste em avaliar o papel de cada variável da base analítica.

Essa auditoria tem como objetivo identificar quais variáveis representam o target, quais funcionam apenas como identificadores ou apoio operacional e quais deverão ser avaliadas quanto à sua elegibilidade como features.

A classificação será realizada antes de qualquer exclusão definitiva, preservando a rastreabilidade das decisões tomadas para a modelagem.

In [0]:
# Objetivo:
#
# Construir um inventário inicial das variáveis
# presentes na base de modelagem.
#
# Justificativa:
#
# Antes de excluir ou selecionar features, é
# necessário registrar todas as variáveis da
# base e permitir sua classificação de acordo
# com o papel desempenhado na modelagem.
#
# Essa etapa torna a seleção posterior das
# features mais clara e auditável.
#
# Ação:
#
# Cria uma tabela com os nomes e tipos de dados
# das variáveis presentes na base.

inventario_variaveis = pd.DataFrame({
    "variavel": base_modelagem.columns,
    "tipo_dado": base_modelagem.dtypes.astype(str).values
})

inventario_variaveis

### 4.1 Classificação das variáveis por papel

Antes da definição das features, as variáveis serão classificadas de acordo com o papel que desempenham no problema de modelagem.

Essa classificação permite separar o target, os identificadores, as variáveis relacionadas à aplicação da avaliação e as informações contextuais incorporadas durante o enriquecimento da base.

A classificação conceitual não representa, neste momento, a decisão definitiva de inclusão ou exclusão de uma variável.

In [0]:
# Objetivo:
#
# Classificar as variáveis da base de acordo
# com seu papel no problema de modelagem.
#
# Justificativa:
#
# A classificação conceitual permite organizar
# as variáveis antes da decisão definitiva sobre
# sua utilização como features.
#
# Essa separação evita confundir o papel de uma
# variável com sua futura inclusão ou exclusão
# do conjunto de preditores.
#
# Ação:
#
# Associa cada variável a uma categoria
# conceitual e adiciona essa informação ao
# inventário de variáveis.

classificacao_variaveis = {
    "nu_ano_avaliacao": "contexto_avaliacao",
    "co_uf": "territorial",
    "sg_uf": "territorial",
    "id_aluno": "identificador",
    "tp_serie": "educacional",
    "id_escola": "identificador",
    "tp_dependencia": "educacional",
    "co_municipio": "territorial",
    "no_municipio": "territorial",
    "in_presenca_lp": "aplicacao_avaliacao",
    "in_preenchimento_lp": "aplicacao_avaliacao",
    "co_caderno_lp": "aplicacao_avaliacao",
    "co_bloco_1": "aplicacao_avaliacao",
    "tx_resposta_bloco_1": "aplicacao_avaliacao",
    "tx_gabarito_bloco_1": "aplicacao_avaliacao",
    "co_bloco_2": "aplicacao_avaliacao",
    "tx_resposta_bloco_2": "aplicacao_avaliacao",
    "tx_gabarito_bloco_2": "aplicacao_avaliacao",
    "co_bloco_3": "aplicacao_avaliacao",
    "tx_resposta_bloco_3": "aplicacao_avaliacao",
    "tx_gabarito_bloco_3": "aplicacao_avaliacao",
    "co_bloco_4": "aplicacao_avaliacao",
    "tx_resposta_bloco_4": "aplicacao_avaliacao",
    "tx_gabarito_bloco_4": "aplicacao_avaliacao",
    "vl_peso_aluno_lp": "aplicacao_avaliacao",
    "vl_proficiencia_lp": "resultado_avaliacao",
    "in_alfabetizado": "target",
    "atlas_idhm": "socioeconomica",
    "atlas_idhm_e": "educacional_contextual",
    "atlas_renda_pc": "socioeconomica",
    "atlas_indice_gini": "socioeconomica",
    "atlas_prop_pobreza_criancas": "socioeconomica",
    "atlas_taxa_criancas_dom_sem_fund": "educacional_contextual",
    "censo_prop_mat_2ano_internet_aprendizagem": "educacional_contextual",
    "censo_prop_mat_2ano_alimentacao": "educacional_contextual",
    "censo_prop_mat_2ano_biblioteca_sala_leitura": "educacional_contextual",
    "fundeb_receita_contribuicao": "financeira_educacional",
    "fundeb_complementacao_uniao": "financeira_educacional",
    "fundeb_receita_total": "financeira_educacional",
    "indicador_meta_final_2025": "educacional_contextual",
    "indicador_pc_aluno_alfabetizado_2024": "educacional_contextual"
}

inventario_variaveis["papel"] = (
    inventario_variaveis["variavel"]
    .map(classificacao_variaveis)
)

inventario_variaveis

### 4.2 Classificação inicial de elegibilidade

Após a classificação conceitual das variáveis, será registrada uma decisão inicial sobre sua elegibilidade para a modelagem.

As variáveis serão classificadas como `manter`, `excluir` ou `investigar`, considerando o momento da predição, o risco de *data leakage*, a função operacional da variável e sua capacidade de generalização.

Nesta etapa, nenhuma coluna será removida da base. O objetivo é documentar as decisões antes da definição final do conjunto de features.

In [0]:
# Objetivo:
#
# Registrar uma decisão inicial de elegibilidade
# para cada variável da base de modelagem.
#
# Justificativa:
#
# A seleção das features deve considerar o
# momento da predição, o risco de data leakage,
# a função da variável e sua capacidade de
# generalização.
#
# Variáveis que ainda dependem de validação
# adicional serão mantidas como investigação,
# evitando decisões prematuras.
#
# Ação:
#
# Classifica cada variável como manter, excluir
# ou investigar, sem remover colunas da base.

decisao_elegibilidade = {
    "nu_ano_avaliacao": "excluir",
    "co_uf": "investigar",
    "sg_uf": "investigar",
    "id_aluno": "excluir",
    "tp_serie": "investigar",
    "id_escola": "investigar",
    "tp_dependencia": "manter",
    "co_municipio": "investigar",
    "no_municipio": "excluir",
    "in_presenca_lp": "excluir",
    "in_preenchimento_lp": "excluir",
    "co_caderno_lp": "excluir",
    "co_bloco_1": "excluir",
    "tx_resposta_bloco_1": "excluir",
    "tx_gabarito_bloco_1": "excluir",
    "co_bloco_2": "excluir",
    "tx_resposta_bloco_2": "excluir",
    "tx_gabarito_bloco_2": "excluir",
    "co_bloco_3": "excluir",
    "tx_resposta_bloco_3": "excluir",
    "tx_gabarito_bloco_3": "excluir",
    "co_bloco_4": "excluir",
    "tx_resposta_bloco_4": "excluir",
    "tx_gabarito_bloco_4": "excluir",
    "vl_peso_aluno_lp": "investigar",
    "vl_proficiencia_lp": "excluir",
    "in_alfabetizado": "excluir",
    "atlas_idhm": "manter",
    "atlas_idhm_e": "manter",
    "atlas_renda_pc": "manter",
    "atlas_indice_gini": "manter",
    "atlas_prop_pobreza_criancas": "manter",
    "atlas_taxa_criancas_dom_sem_fund": "manter",
    "censo_prop_mat_2ano_internet_aprendizagem": "manter",
    "censo_prop_mat_2ano_alimentacao": "manter",
    "censo_prop_mat_2ano_biblioteca_sala_leitura": "manter",
    "fundeb_receita_contribuicao": "manter",
    "fundeb_complementacao_uniao": "manter",
    "fundeb_receita_total": "manter",
    "indicador_meta_final_2025": "investigar",
    "indicador_pc_aluno_alfabetizado_2024": "manter"
}

inventario_variaveis["decisao_inicial"] = (
    inventario_variaveis["variavel"]
    .map(decisao_elegibilidade)
)

inventario_variaveis

### 4.3 Validação da classificação inicial

Após registrar a elegibilidade inicial das variáveis, é necessário validar se todas as colunas receberam uma decisão e verificar sua distribuição entre as categorias definidas.

Essa conferência garante que nenhuma variável permaneça sem classificação antes da análise específica dos casos marcados como `investigar`.

In [0]:
# Objetivo:
#
# Validar a classificação inicial de elegibilidade
# das variáveis da base de modelagem.
#
# Justificativa:
#
# Antes de analisar individualmente as variáveis
# classificadas como investigar, é necessário
# confirmar que todas as colunas receberam uma
# decisão inicial.
#
# A contagem por categoria também permite verificar
# a distribuição das variáveis entre manter,
# excluir e investigar.
#
# Ação:
#
# Conta as variáveis em cada decisão e verifica
# se alguma permaneceu sem classificação.

validacao_elegibilidade = pd.Series({
    "total_variaveis": len(inventario_variaveis),
    "manter": (inventario_variaveis["decisao_inicial"] == "manter").sum(),
    "excluir": (inventario_variaveis["decisao_inicial"] == "excluir").sum(),
    "investigar": (inventario_variaveis["decisao_inicial"] == "investigar").sum(),
    "sem_classificacao": inventario_variaveis["decisao_inicial"].isna().sum()
})

validacao_elegibilidade

### 4.4 Conclusão da auditoria das variáveis

A auditoria das variáveis inicialmente classificadas como `investigar` foi concluída com base nas evidências observadas na base, no dicionário de dados, no LeiaMe dos microdados da AEEB 2025 e nas definições metodológicas adotadas para o problema preditivo.

As decisões consideram a variabilidade das colunas, redundâncias, cardinalidade, função operacional, risco de memorização e disponibilidade da informação no momento definido para a predição.

| Variável | Evidência considerada | Decisão |
|---|---|---|
| `co_uf` | Possui 27 valores únicos e apresenta correspondência um-para-um com `sg_uf`, representando a mesma informação territorial em formato codificado. | Manter como representação da unidade da federação. |
| `sg_uf` | Possui os mesmos 27 domínios de `co_uf` e apresenta correspondência um-para-um com o código da UF. | Excluir por redundância com `co_uf`. |
| `tp_serie` | Apresenta apenas um valor único em toda a base, não oferecendo variabilidade para discriminar os registros. | Excluir por constância. |
| `id_escola` | O dicionário e o LeiaMe informam que o código real da escola foi substituído por uma máscara. Além disso, a variável apresenta alta cardinalidade. | Excluir como feature e preservar apenas como variável auxiliar para eventual agrupamento. |
| `co_municipio` | Identifica o município e apresenta alta cardinalidade. Como diversas features enriquecidas possuem granularidade municipal, seu uso direto como preditor pode favorecer memorização territorial. | Excluir como feature e preservar como variável auxiliar para agrupamento e validação. |
| `vl_peso_aluno_lp` | É definido como o peso do aluno na prova de Língua Portuguesa, porém a documentação consultada não permite comprovar sua disponibilidade antes da aplicação da avaliação. | Excluir por disponibilidade temporal não comprovada no cenário preventivo. |
| `indicador_meta_final_2025` | Representa a meta municipal estabelecida para o ano de 2025 e não o resultado observado da avaliação. | Manter como feature, considerando a premissa de que a meta estava definida antes da divulgação dos resultados de 2025. |

A investigação também confirmou que `co_uf` e `sg_uf` carregam a mesma informação territorial, enquanto `tp_serie` é constante em toda a base.

Para `id_aluno`, `id_escola` e `co_municipio`, a exclusão do conjunto de features não implica sua remoção completa do processo de modelagem.

O `id_aluno` será preservado exclusivamente como chave de rastreabilidade, permitindo associar posteriormente cada previsão à observação original sem oferecer ao modelo uma informação de identificação como preditor.

As variáveis `id_escola` e `co_municipio` também serão mantidas separadamente como variáveis auxiliares para apoiar agrupamento, validação territorial e análises posteriores.

No caso de `vl_peso_aluno_lp`, a exclusão não decorre de *data leakage* comprovado, mas da ausência de evidência documental suficiente para garantir sua disponibilidade no momento da predição.

Com essas decisões, todas as variáveis inicialmente classificadas como `investigar` passam a possuir uma função definida para a próxima etapa da preparação dos dados.


### 4.5 Consolidação da decisão final das variáveis

Após a conclusão da auditoria, as decisões referentes às variáveis inicialmente classificadas como `investigar` serão consolidadas no inventário.

A classificação inicial será preservada para manter a rastreabilidade do processo, enquanto uma nova coluna registrará a decisão final adotada para cada variável.

As variáveis `id_aluno`, `id_escola` e `co_municipio`, embora excluídas do conjunto de features, serão preservadas separadamente como variáveis auxiliares.

O `id_aluno` será mantido exclusivamente para garantir a rastreabilidade das previsões até a observação original, enquanto `id_escola` e `co_municipio` permanecerão disponíveis para agrupamento, validação e análises posteriores.


In [0]:
# Objetivo:
#
# Consolidar as decisões finais da auditoria
# das variáveis para modelagem.
#
# Justificativa:
#
# A classificação inicial deve ser preservada
# para manter a rastreabilidade das decisões
# tomadas durante a auditoria.
#
# As variáveis inicialmente classificadas como
# investigar já possuem uma decisão fundamentada
# pelas evidências analisadas.
#
# Ação:
#
# Cria a coluna decisao_final a partir da decisão
# inicial e atualiza as variáveis investigadas
# conforme a conclusão da auditoria.

inventario_variaveis["decisao_final"] = (
    inventario_variaveis["decisao_inicial"]
)

decisoes_finais = {
    "co_uf": "manter",
    "sg_uf": "excluir",
    "tp_serie": "excluir",
    "id_escola": "excluir",
    "co_municipio": "excluir",
    "vl_peso_aluno_lp": "excluir",
    "indicador_meta_final_2025": "manter"
}

inventario_variaveis["decisao_final"] = (
    inventario_variaveis["variavel"]
    .map(decisoes_finais)
    .fillna(inventario_variaveis["decisao_final"])
)

inventario_variaveis

### 4.6 Validação da decisão final

Após a consolidação das decisões, será realizada uma validação final do inventário para confirmar que todas as variáveis possuem uma classificação definitiva.

Essa conferência encerra a auditoria das variáveis e garante que nenhuma coluna permaneça como `investigar` ou sem decisão antes da construção dos conjuntos utilizados na modelagem.

In [0]:
# Objetivo:
#
# Validar a classificação final das variáveis
# após a conclusão da auditoria.
#
# Justificativa:
#
# Antes da construção dos conjuntos de modelagem,
# é necessário confirmar que todas as variáveis
# possuem uma decisão definitiva.
#
# A validação também permite reconciliar o total
# de variáveis entre as categorias manter e
# excluir.
#
# Ação:
#
# Conta as decisões finais e verifica se existem
# variáveis ainda classificadas como investigar
# ou sem decisão final.

validacao_decisao_final = pd.Series({
    "total_variaveis": len(inventario_variaveis),
    "manter": (
        inventario_variaveis["decisao_final"] == "manter"
    ).sum(),
    "excluir": (
        inventario_variaveis["decisao_final"] == "excluir"
    ).sum(),
    "investigar": (
        inventario_variaveis["decisao_final"] == "investigar"
    ).sum(),
    "sem_decisao_final": (
        inventario_variaveis["decisao_final"].isna()
    ).sum()
})

validacao_decisao_final

## 5. Construção dos conjuntos para modelagem

Com a auditoria das variáveis concluída, serão construídas as estruturas que servirão de base para as próximas etapas da modelagem.

O conjunto de features (`X`) será formado exclusivamente pelas variáveis classificadas como `manter` na decisão final, enquanto a variável `in_alfabetizado` será utilizada como target (`y`).

As variáveis `id_aluno`, `id_escola` e `co_municipio` não serão utilizadas como features, mas serão preservadas separadamente como variáveis auxiliares.

O `id_aluno` será utilizado exclusivamente para rastreabilidade das previsões, enquanto `id_escola` e `co_municipio` permanecerão disponíveis para agrupamento, validação e análises posteriores.

Nenhuma imputação, transformação ou codificação será aplicada nesta etapa. Esses procedimentos serão realizados posteriormente dentro da pipeline de Machine Learning, após a definição da estratégia de separação dos dados.


In [0]:
# Objetivo:
#
# Construir os conjuntos de features, target
# e variáveis auxiliares para modelagem.
#
# Justificativa:
#
# A auditoria definiu quais variáveis podem ser
# utilizadas como features e quais devem permanecer
# fora do conjunto preditivo.
#
# As variáveis de aluno, escola e município serão
# preservadas separadamente para garantir
# rastreabilidade, agrupamento e apoio às etapas
# posteriores de validação.
#
# Essas variáveis auxiliares não serão utilizadas
# como features pelos modelos.
#
# Ação:
#
# Obtém as features aprovadas a partir do inventário,
# separa o target e preserva as variáveis auxiliares.

features_modelagem = (
    inventario_variaveis.loc[
        inventario_variaveis["decisao_final"] == "manter",
        "variavel"
    ]
    .tolist()
)

X = base_modelagem[features_modelagem].copy()

y = base_modelagem["in_alfabetizado"].copy()

variaveis_auxiliares = (
    base_modelagem[
        [
            "id_aluno",
            "id_escola",
            "co_municipio"
        ]
    ]
    .copy()
)

X.shape, y.shape, variaveis_auxiliares.shape


### 5.1 Estratégia de separação dos dados

A separação entre treino, validação e teste deve considerar a estrutura hierárquica da base e a granularidade das variáveis utilizadas na modelagem.

Embora cada registro represente um aluno, parte das features possui granularidade municipal e, portanto, apresenta valores compartilhados entre alunos pertencentes ao mesmo município.

Uma divisão puramente aleatória por aluno poderia distribuir registros de um mesmo município entre diferentes conjuntos, permitindo que contextos territoriais já observados durante o treinamento também estivessem presentes na validação e no teste.

Para reduzir esse risco e produzir uma avaliação mais rigorosa da capacidade de generalização, será adotada uma estratégia de separação por grupos, utilizando `co_municipio` como variável auxiliar.

Dessa forma, os municípios utilizados no treinamento não deverão aparecer nos conjuntos de validação e teste.

A variável `co_municipio` será utilizada exclusivamente para controlar a separação dos dados e permanecerá fora do conjunto de features (`X`).

As variáveis auxiliares `id_aluno`, `id_escola` e `co_municipio` acompanharão os mesmos índices produzidos pelo `GroupShuffleSplit`. Isso preserva o alinhamento entre cada linha de features, seu target e suas chaves de rastreabilidade, sem inserir esses identificadores no treinamento dos modelos.


In [0]:
# Objetivo:
#
# Separar os dados em conjuntos de treino,
# validação e teste utilizando o município
# como unidade de agrupamento.
#
# Justificativa:
#
# Parte das features possui granularidade municipal.
# Por isso, alunos de um mesmo município não devem
# ser distribuídos entre conjuntos diferentes.
#
# Registros sem identificação de município não podem
# participar adequadamente da separação por grupos.
# Esses casos serão removidos apenas da população
# destinada à modelagem, sem imputação artificial
# da variável de agrupamento.
#
# As variáveis auxiliares devem acompanhar exatamente
# os mesmos índices dos conjuntos preditivos para
# preservar a rastreabilidade das observações.
#
# Ação:
#
# Remove os registros sem município e realiza duas
# separações com GroupShuffleSplit: primeiro treino
# e temporário; depois validação e teste. As variáveis
# auxiliares são particionadas pelos mesmos índices.

from sklearn.model_selection import GroupShuffleSplit

mascara_municipio_valido = (
    variaveis_auxiliares["co_municipio"].notna()
)

X_modelagem = (
    X.loc[mascara_municipio_valido]
    .copy()
)

y_modelagem = (
    y.loc[mascara_municipio_valido]
    .copy()
)

auxiliares_modelagem = (
    variaveis_auxiliares.loc[mascara_municipio_valido]
    .copy()
)

grupos_modelagem = (
    auxiliares_modelagem["co_municipio"]
    .copy()
)

split_treino = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42
)

indice_treino, indice_temporario = next(
    split_treino.split(
        X_modelagem,
        y_modelagem,
        groups=grupos_modelagem
    )
)

X_treino = X_modelagem.iloc[indice_treino].copy()
X_temporario = X_modelagem.iloc[indice_temporario].copy()

y_treino = y_modelagem.iloc[indice_treino].copy()
y_temporario = y_modelagem.iloc[indice_temporario].copy()

auxiliares_treino = (
    auxiliares_modelagem.iloc[indice_treino]
    .copy()
)

auxiliares_temporario = (
    auxiliares_modelagem.iloc[indice_temporario]
    .copy()
)

grupos_temporario = (
    auxiliares_temporario["co_municipio"]
)

split_validacao_teste = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

indice_validacao, indice_teste = next(
    split_validacao_teste.split(
        X_temporario,
        y_temporario,
        groups=grupos_temporario
    )
)

X_validacao = (
    X_temporario.iloc[indice_validacao]
    .copy()
)

X_teste = (
    X_temporario.iloc[indice_teste]
    .copy()
)

y_validacao = (
    y_temporario.iloc[indice_validacao]
    .copy()
)

y_teste = (
    y_temporario.iloc[indice_teste]
    .copy()
)

auxiliares_validacao = (
    auxiliares_temporario.iloc[indice_validacao]
    .copy()
)

auxiliares_teste = (
    auxiliares_temporario.iloc[indice_teste]
    .copy()
)

X_treino.shape, X_validacao.shape, X_teste.shape


In [0]:
# Objetivo:
#
# Validar a separação dos dados realizada
# por agrupamento municipal e o alinhamento
# das variáveis auxiliares.
#
# Justificativa:
#
# A estratégia definida exige que municípios
# presentes em um conjunto não apareçam nos
# demais, garantindo independência territorial
# entre treino, validação e teste.
#
# Também é necessário confirmar que features,
# target e variáveis auxiliares preservam os
# mesmos registros e índices em cada partição.
#
# Ação:
#
# Recupera os municípios de cada conjunto,
# verifica possíveis interseções e valida o
# alinhamento das estruturas auxiliares.

grupos_treino = set(
    auxiliares_treino["co_municipio"]
)

grupos_validacao = set(
    auxiliares_validacao["co_municipio"]
)

grupos_teste = set(
    auxiliares_teste["co_municipio"]
)

validacao_split = pd.Series({
    "registros_modelagem": len(X_modelagem),
    "registros_treino": len(X_treino),
    "registros_validacao": len(X_validacao),
    "registros_teste": len(X_teste),
    "municipios_treino": len(grupos_treino),
    "municipios_validacao": len(grupos_validacao),
    "municipios_teste": len(grupos_teste),
    "intersecao_treino_validacao":
        len(grupos_treino & grupos_validacao),
    "intersecao_treino_teste":
        len(grupos_treino & grupos_teste),
    "intersecao_validacao_teste":
        len(grupos_validacao & grupos_teste),
    "auxiliares_treino_alinhados": (
        X_treino.index.equals(auxiliares_treino.index)
        and y_treino.index.equals(auxiliares_treino.index)
    ),
    "auxiliares_validacao_alinhados": (
        X_validacao.index.equals(auxiliares_validacao.index)
        and y_validacao.index.equals(auxiliares_validacao.index)
    ),
    "auxiliares_teste_alinhados": (
        X_teste.index.equals(auxiliares_teste.index)
        and y_teste.index.equals(auxiliares_teste.index)
    )
})

validacao_split


## 6. Definição da estratégia de pré-processamento

Com os conjuntos de treino, validação e teste definidos e territorialmente independentes, inicia-se a preparação da estratégia de pré-processamento que será posteriormente integrada à pipeline de Machine Learning.

Nesta etapa, serão analisadas as características das features presentes no conjunto de treino, incluindo seus tipos de dados, cardinalidade e ocorrência de valores ausentes.

Essa análise permitirá definir tratamentos distintos para variáveis numéricas e categóricas, além das estratégias de imputação e transformação adequadas a cada grupo.

As decisões serão realizadas exclusivamente a partir dos dados de treino, preservando os conjuntos de validação e teste para as etapas posteriores de avaliação dos modelos.

In [0]:
# Objetivo:
#
# Caracterizar as features utilizadas na
# modelagem a partir do conjunto de treino.
#
# Justificativa:
#
# A definição da estratégia de pré-processamento
# depende do tipo de dado, da cardinalidade e da
# presença de valores ausentes em cada feature.
#
# Essa análise deve utilizar somente o conjunto
# de treino, preservando validação e teste para
# a avaliação posterior dos modelos.
#
# Ação:
#
# Resume o tipo de dado, a quantidade de valores
# únicos e os valores ausentes das features
# presentes no conjunto de treino.

resumo_pre_processamento = pd.DataFrame({
    "variavel": X_treino.columns,
    "tipo_dado": X_treino.dtypes.astype(str).values,
    "valores_unicos": X_treino.nunique().values,
    "valores_ausentes": X_treino.isna().sum().values
})

resumo_pre_processamento

### 6.1 Análise do padrão de valores ausentes

Antes da definição da estratégia de imputação, será analisado o padrão de ocorrência dos valores ausentes nas diferentes famílias de features.

As variáveis provenientes de uma mesma fonte apresentam quantidades idênticas de valores ausentes, o que pode indicar indisponibilidade conjunta da informação contextual para determinados registros.

A identificação desse padrão é relevante para decidir se a imputação isolada dos valores é suficiente ou se a ausência da informação deve também ser representada explicitamente durante o pré-processamento.

A análise será realizada exclusivamente sobre o conjunto de treino.

In [0]:
# Objetivo:
#
# Verificar se os valores ausentes das features
# pertencentes a uma mesma família ocorrem nos
# mesmos registros do conjunto de treino.
#
# Justificativa:
#
# Ausências coincidentes podem representar a
# indisponibilidade conjunta de uma fonte de
# informação contextual.
#
# Esse padrão deve ser conhecido antes da escolha
# da estratégia de imputação utilizada na pipeline.
#
# Ação:
#
# Compara as máscaras de valores ausentes das
# variáveis pertencentes às famílias Atlas,
# Censo Escolar, FUNDEB e indicadores.

familias_features = {
    "atlas": [
        "atlas_idhm",
        "atlas_idhm_e",
        "atlas_renda_pc",
        "atlas_indice_gini",
        "atlas_prop_pobreza_criancas",
        "atlas_taxa_criancas_dom_sem_fund"
    ],
    "censo": [
        "censo_prop_mat_2ano_internet_aprendizagem",
        "censo_prop_mat_2ano_alimentacao",
        "censo_prop_mat_2ano_biblioteca_sala_leitura"
    ],
    "fundeb": [
        "fundeb_receita_contribuicao",
        "fundeb_complementacao_uniao",
        "fundeb_receita_total"
    ],
    "indicadores": [
        "indicador_meta_final_2025",
        "indicador_pc_aluno_alfabetizado_2024"
    ]
}

resultado_ausencias = {}

for familia, colunas in familias_features.items():

    mascara_referencia = (
        X_treino[colunas[0]].isna()
    )

    ausencias_coincidem = all(
        X_treino[coluna]
        .isna()
        .equals(mascara_referencia)
        for coluna in colunas[1:]
    )

    resultado_ausencias[familia] = {
        "registros_ausentes": mascara_referencia.sum(),
        "ausencias_coincidem": ausencias_coincidem
    }

pd.DataFrame(resultado_ausencias).T

### 6.2 Estratégia de tratamento dos valores ausentes

A análise do conjunto de treino demonstrou que os valores ausentes apresentam um padrão estruturado por fonte de dados.

Dentro de cada uma das famílias Atlas, Censo Escolar, FUNDEB e Indicadores, as ausências ocorrem exatamente nos mesmos registros. Dessa forma, a indisponibilidade não será tratada apenas como ausência isolada de uma variável, mas também como uma característica do conjunto de informações provenientes de cada fonte.

Para as features numéricas, será utilizada imputação pela mediana. Essa estratégia oferece maior robustez à presença de distribuições assimétricas e valores extremos, especialmente em variáveis socioeconômicas e financeiras.

As medianas serão aprendidas exclusivamente a partir do conjunto de treino e aplicadas posteriormente aos conjuntos de validação e teste por meio da pipeline de pré-processamento.

Além da imputação, serão criados quatro indicadores binários de ausência, correspondentes às famílias:

- `atlas_dados_ausentes`;
- `censo_dados_ausentes`;
- `fundeb_dados_ausentes`;
- `indicadores_dados_ausentes`.

Como as máscaras de ausência são idênticas entre as variáveis pertencentes a uma mesma família, será utilizado apenas um indicador por fonte, evitando a criação de atributos redundantes.

Esses indicadores permitirão preservar a informação de que determinado conjunto de dados contextuais estava originalmente indisponível, enquanto os valores numéricos ausentes serão tratados pela estratégia de imputação definida.

### 6.3 Estratégia de transformação das variáveis

As features utilizadas na modelagem apresentam naturezas e escalas distintas e, portanto, exigem estratégias específicas de transformação.

As variáveis `co_uf` e `tp_dependencia`, embora armazenadas numericamente, representam categorias nominais. Essas features serão tratadas como categóricas e codificadas por meio de `OneHotEncoder`.

O encoder será configurado para permitir categorias não observadas durante o treinamento, evitando falhas quando novas categorias estiverem presentes nos conjuntos de validação ou teste.

As demais features serão tratadas como numéricas e terão seus valores ausentes imputados pela mediana, conforme definido anteriormente.

A aplicação de padronização será condicionada ao algoritmo utilizado. Modelos sensíveis à escala, como modelos lineares, utilizarão `StandardScaler` após a imputação das variáveis numéricas.

Para modelos baseados em árvores, a padronização não será aplicada, pois suas decisões são baseadas em pontos de corte das features e não dependem da comparação direta entre suas escalas.

Dessa forma, o projeto utilizará uma arquitetura de pré-processamento comum para imputação e codificação, permitindo adaptações específicas de transformação de acordo com a família do modelo avaliado.

### 6.4 Construção da arquitetura de pré-processamento

A estratégia definida nas etapas anteriores será implementada por meio de componentes compatíveis com a API do Scikit-learn, permitindo sua posterior integração à pipeline completa de Machine Learning.

A engenharia de atributos será responsável pela criação dos indicadores de ausência das famílias Atlas, Censo Escolar, FUNDEB e Indicadores.

Em seguida, o pré-processamento será dividido em dois fluxos:

- variáveis categóricas: codificação por `OneHotEncoder`;
- variáveis numéricas: imputação dos valores ausentes pela mediana.

A padronização das variáveis numéricas não será incorporada ao pré-processamento comum neste momento, pois sua utilização dependerá da família do modelo avaliado.

Toda a arquitetura será definida sem realizar ajuste sobre os dados nesta etapa. O aprendizado dos parâmetros de imputação, codificação e demais transformações ocorrerá posteriormente utilizando exclusivamente o conjunto de treino.

In [0]:
# Objetivo:
#
# Construir os componentes automatizados de
# engenharia de atributos e pré-processamento.
#
# Justificativa:
#
# O pré-processamento deve permanecer integrado
# ao fluxo de Machine Learning para garantir
# reprodutibilidade e evitar data leakage.
#
# Os indicadores de ausência serão criados por
# fonte de dados, enquanto variáveis numéricas
# e categóricas receberão tratamentos distintos.
#
# Ação:
#
# Define um transformer para engenharia dos
# indicadores de ausência e constrói pipelines
# numérica e categórica integradas a um
# ColumnTransformer.

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


class IndicadoresAusencia(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X_transformado = X.copy()

        X_transformado["atlas_dados_ausentes"] = (
            X_transformado["atlas_idhm"].isna().astype(int)
        )

        X_transformado["censo_dados_ausentes"] = (
            X_transformado[
                "censo_prop_mat_2ano_internet_aprendizagem"
            ]
            .isna()
            .astype(int)
        )

        X_transformado["fundeb_dados_ausentes"] = (
            X_transformado[
                "fundeb_receita_contribuicao"
            ]
            .isna()
            .astype(int)
        )

        X_transformado["indicadores_dados_ausentes"] = (
            X_transformado[
                "indicador_meta_final_2025"
            ]
            .isna()
            .astype(int)
        )

        return X_transformado


features_categoricas = [
    "co_uf",
    "tp_dependencia"
]

features_numericas = [
    coluna
    for coluna in X_treino.columns
    if coluna not in features_categoricas
]

features_indicadores = [
    "atlas_dados_ausentes",
    "censo_dados_ausentes",
    "fundeb_dados_ausentes",
    "indicadores_dados_ausentes"
]


pipeline_numerica = Pipeline(
    steps=[
        (
            "imputacao",
            SimpleImputer(strategy="median")
        )
    ]
)


pipeline_categorica = Pipeline(
    steps=[
        (
            "encoding",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


pre_processamento = ColumnTransformer(
    transformers=[
        (
            "numericas",
            pipeline_numerica,
            features_numericas
        ),
        (
            "categoricas",
            pipeline_categorica,
            features_categoricas
        ),
        (
            "indicadores_ausencia",
            "passthrough",
            features_indicadores
        )
    ]
)


pipeline_preparacao = Pipeline(
    steps=[
        (
            "engenharia_atributos",
            IndicadoresAusencia()
        ),
        (
            "pre_processamento",
            pre_processamento
        )
    ]
)

pipeline_preparacao

### 6.5 Validação da arquitetura de pré-processamento

Antes de integrar o pré-processamento aos modelos, será realizada uma validação técnica de sua arquitetura.

O objetivo desta etapa é verificar se a engenharia dos indicadores de ausência, a imputação das variáveis numéricas e a codificação das variáveis categóricas funcionam de forma integrada.

Para essa verificação será utilizada apenas uma pequena amostra do conjunto de treino. Essa amostra possui finalidade exclusivamente técnica e não será utilizada para treinamento ou avaliação dos modelos.

A arquitetura definitiva permanecerá sem ajuste prévio, pois seu treinamento será realizado posteriormente como parte da pipeline completa de Machine Learning.

In [0]:
# Objetivo:
#
# Validar tecnicamente a arquitetura de
# pré-processamento definida.
#
# Justificativa:
#
# Antes de integrar o pré-processamento aos
# modelos, é necessário verificar se todos os
# componentes funcionam corretamente em conjunto.
#
# A validação utilizará apenas uma pequena amostra
# do conjunto de treino e não será empregada na
# modelagem ou avaliação dos algoritmos.
#
# Ação:
#
# Cria uma cópia independente da pipeline,
# ajusta essa cópia sobre uma pequena amostra
# de treino e verifica a dimensão resultante.

from sklearn.base import clone

amostra_validacao_pipeline = (
    X_treino
    .sample(
        n=10000,
        random_state=42
    )
)

pipeline_teste = clone(
    pipeline_preparacao
)

dados_transformados_teste = (
    pipeline_teste
    .fit_transform(amostra_validacao_pipeline)
)

pd.Series({
    "registros_entrada": len(amostra_validacao_pipeline),
    "features_entrada": amostra_validacao_pipeline.shape[1],
    "registros_saida": dados_transformados_teste.shape[0],
    "features_saida": dados_transformados_teste.shape[1]
})

## 7. Persistência dos conjuntos para modelagem

Os conjuntos de treino, validação e teste definidos neste notebook serão persistidos para permitir sua reutilização nas etapas posteriores de modelagem.

A persistência evita a necessidade de repetir o processo de separação dos dados e garante que os modelos sejam avaliados utilizando exatamente as mesmas partições territoriais previamente definidas e validadas.

Os conjuntos preditivos serão armazenados antes da aplicação definitiva das transformações de pré-processamento. Dessa forma, imputação, codificação e demais transformações continuarão sendo aprendidas exclusivamente a partir dos dados de treino quando integradas às pipelines completas de Machine Learning.

As variáveis auxiliares `id_aluno`, `id_escola` e `co_municipio` serão persistidas em arquivos Parquet separados, mantendo o mesmo alinhamento das partições de treino, validação e teste. Essa separação preserva a rastreabilidade sem permitir que identificadores sejam utilizados acidentalmente como features pelos modelos.

Será utilizado o formato Parquet, que preserva os tipos de dados e oferece armazenamento eficiente para conjuntos de grande volume.

A lógica reutilizável de engenharia de atributos e pré-processamento será mantida separada dos dados e disponibilizada para a etapa de modelagem por meio de um módulo Python específico.


In [0]:
# Objetivo:
#
# Persistir os conjuntos de treino, validação
# e teste definidos para a modelagem, além das
# respectivas variáveis auxiliares.
#
# Justificativa:
#
# A persistência das partições garante que as
# etapas posteriores utilizem exatamente os
# mesmos conjuntos territoriais já definidos
# e validados neste notebook.
#
# Os identificadores serão mantidos em arquivos
# auxiliares separados para preservar a
# rastreabilidade sem incluí-los nas features.
#
# Os dados preditivos serão armazenados antes das
# etapas definitivas de imputação e transformação,
# que permanecerão integradas às pipelines
# completas de Machine Learning.
#
# Ação:
#
# Reúne features e target de cada partição,
# salva os conjuntos preditivos em Parquet e
# persiste separadamente os metadados auxiliares
# correspondentes às mesmas linhas.

treino = X_treino.copy()

treino["in_alfabetizado"] = (
    y_treino
)

validacao = X_validacao.copy()

validacao["in_alfabetizado"] = (
    y_validacao
)

teste = X_teste.copy()

teste["in_alfabetizado"] = (
    y_teste
)

# Remove o índice antes da persistência, mantendo
# o alinhamento entre os arquivos pela mesma ordem
# de linhas produzida pelo split.
treino = treino.reset_index(drop=True)
validacao = validacao.reset_index(drop=True)
teste = teste.reset_index(drop=True)

auxiliares_treino_persistencia = (
    auxiliares_treino.reset_index(drop=True)
)

auxiliares_validacao_persistencia = (
    auxiliares_validacao.reset_index(drop=True)
)

auxiliares_teste_persistencia = (
    auxiliares_teste.reset_index(drop=True)
)

treino.to_parquet(
    f"{MODELAGEM_PATH}/treino.parquet",
    index=False
)

validacao.to_parquet(
    f"{MODELAGEM_PATH}/validacao.parquet",
    index=False
)

teste.to_parquet(
    f"{MODELAGEM_PATH}/teste.parquet",
    index=False
)

auxiliares_treino_persistencia.to_parquet(
    f"{MODELAGEM_PATH}/auxiliares_treino.parquet",
    index=False
)

auxiliares_validacao_persistencia.to_parquet(
    f"{MODELAGEM_PATH}/auxiliares_validacao.parquet",
    index=False
)

auxiliares_teste_persistencia.to_parquet(
    f"{MODELAGEM_PATH}/auxiliares_teste.parquet",
    index=False
)

pd.Series({
    "registros_treino": len(treino),
    "registros_validacao": len(validacao),
    "registros_teste": len(teste),
    "colunas_treino": treino.shape[1],
    "colunas_validacao": validacao.shape[1],
    "colunas_teste": teste.shape[1],
    "colunas_auxiliares": auxiliares_treino_persistencia.shape[1]
})


### 7.1 Validação dos arquivos persistidos

Após a gravação dos conjuntos de treino, validação e teste, será realizada uma leitura de verificação dos arquivos persistidos.

Essa etapa permite confirmar que as dimensões, o conjunto de colunas e a variável-alvo foram preservados corretamente no formato Parquet.

Também serão validados os três arquivos auxiliares de rastreabilidade, verificando a presença de `id_aluno`, `id_escola` e `co_municipio` e a correspondência da quantidade de linhas com cada partição preditiva.

A validação garante que o Notebook 08 poderá carregar os mesmos artefatos sem depender da memória da sessão utilizada neste notebook e sem misturar identificadores às features do modelo.


In [0]:
# Objetivo:
#
# Validar os arquivos Parquet persistidos para
# as etapas posteriores de modelagem e os arquivos
# auxiliares de rastreabilidade.
#
# Justificativa:
#
# A leitura de verificação confirma que os
# conjuntos foram gravados corretamente e que
# poderão ser reutilizados no próximo notebook
# sem perda de estrutura ou identificação.
#
# Ação:
#
# Carrega os arquivos persistidos e valida suas
# dimensões, presença do target, presença das
# chaves auxiliares e correspondência de registros.

treino_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/treino.parquet"
)

validacao_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/validacao.parquet"
)

teste_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/teste.parquet"
)

auxiliares_treino_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_treino.parquet"
)

auxiliares_validacao_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_validacao.parquet"
)

auxiliares_teste_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_teste.parquet"
)

colunas_auxiliares_esperadas = {
    "id_aluno",
    "id_escola",
    "co_municipio"
}

validacao_persistencia = pd.Series({
    "treino_registros": treino_validacao.shape[0],
    "treino_colunas": treino_validacao.shape[1],
    "validacao_registros": validacao_validacao.shape[0],
    "validacao_colunas": validacao_validacao.shape[1],
    "teste_registros": teste_validacao.shape[0],
    "teste_colunas": teste_validacao.shape[1],
    "target_treino": "in_alfabetizado" in treino_validacao.columns,
    "target_validacao": "in_alfabetizado" in validacao_validacao.columns,
    "target_teste": "in_alfabetizado" in teste_validacao.columns,
    "auxiliares_treino_ok": (
        set(auxiliares_treino_validacao.columns)
        == colunas_auxiliares_esperadas
        and len(auxiliares_treino_validacao)
        == len(treino_validacao)
    ),
    "auxiliares_validacao_ok": (
        set(auxiliares_validacao_validacao.columns)
        == colunas_auxiliares_esperadas
        and len(auxiliares_validacao_validacao)
        == len(validacao_validacao)
    ),
    "auxiliares_teste_ok": (
        set(auxiliares_teste_validacao.columns)
        == colunas_auxiliares_esperadas
        and len(auxiliares_teste_validacao)
        == len(teste_validacao)
    )
})

validacao_persistencia


## 8. Conclusão da preparação para modelagem

A etapa de preparação para modelagem foi concluída com a definição de uma estrutura reproduzível e compatível com a construção das pipelines completas de Machine Learning.

Ao longo deste notebook, as variáveis da base analítica foram auditadas quanto ao seu papel e à sua elegibilidade para utilização no modelo, resultando em 16 features aprovadas para o conjunto preditivo.

O target `in_alfabetizado` foi separado das features. As variáveis `id_aluno`, `id_escola` e `co_municipio` foram mantidas fora do conjunto preditivo e preservadas separadamente como variáveis auxiliares: `id_aluno` para rastreabilidade das previsões e `id_escola`/`co_municipio` para apoio a agrupamentos, validações e análises posteriores.

A separação dos dados foi realizada por agrupamento municipal, garantindo que um mesmo município não estivesse simultaneamente presente nos conjuntos de treino, validação e teste. A validação confirmou ausência de interseção entre os municípios das três partições e o alinhamento entre features, target e variáveis auxiliares.

Os registros sem identificação de município foram excluídos exclusivamente da população destinada à modelagem, pois não poderiam ser atribuídos de forma segura aos grupos utilizados na separação.

A análise dos valores ausentes identificou padrões estruturados por fonte de dados. Com base nesse comportamento, foi definida a imputação pela mediana para as variáveis numéricas, acompanhada da criação de indicadores de ausência para as famílias Atlas, Censo Escolar, FUNDEB e Indicadores.

As variáveis `co_uf` e `tp_dependencia` foram tratadas como categóricas, com estratégia de transformação por meio de `OneHotEncoder`.

A engenharia de atributos, a imputação e a codificação foram encapsuladas em componentes compatíveis com a API do Scikit-learn. A validação técnica da arquitetura confirmou a preservação dos registros e a transformação das 16 features originais em 47 features na amostra utilizada para verificação.

Para garantir a continuidade reproduzível do projeto, os conjuntos de treino, validação e teste foram persistidos em formato Parquet no diretório `tech_challenge_fase3/modelagem`, contendo as 16 features selecionadas e o target. Em arquivos Parquet auxiliares separados foram persistidos `id_aluno`, `id_escola` e `co_municipio`, mantendo a mesma ordem de registros de cada partição e preservando a rastreabilidade sem introduzir identificadores nas features.

A lógica reutilizável de engenharia de atributos e pré-processamento foi externalizada para o módulo `src/preprocessing.py`, permitindo sua utilização nas etapas posteriores sem duplicação de código e mantendo separadas as responsabilidades de preparação dos dados e modelagem.

Os artefatos foram mantidos antes da aplicação definitiva das transformações. Dessa forma, os parâmetros de imputação, codificação e, quando aplicável, padronização serão aprendidos exclusivamente a partir dos dados de treino quando o pré-processamento for integrado aos estimadores.

A próxima etapa será dedicada à construção, treinamento, validação e comparação das pipelines completas de Machine Learning, contemplando modelos de referência, métricas adequadas ao problema, análise de generalização, validação estatística, otimização e interpretabilidade dos resultados.
